In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError, IntegrityError
from pandas.errors import DatabaseError
load_dotenv()

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

engine = create_engine(f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

In [ ]:

        
def extract(customers_path, orders_path):
    try:
        customers_df = pd.read_csv(customers_path)
    except FileNotFoundError:
        print('Customers file not found')
        customers_df = None
    try:   
            orders_df = pd.read_csv(orders_path)
    except FileNotFoundError:
            print('Orders file not found')
            orders_df = None
    return customers_df, orders_df

In [ ]:
def clean(customers_df, orders_df):
    if customers_df is None or orders_df is None:
        return None, None, None
    else:
        customers_df['city'] = customers_df['city'].str.strip().str.title()
        customers_df['city'] = customers_df['city'].replace({"Ph": "Port Harcourt"})
        customers_df['customer_name'] = customers_df['customer_name'].str.strip().str.title()
        customers_df = customers_df.drop_duplicates()
        customers_df['email_missing'] = customers_df['email'].isnull()
        orders_df['unit_price'] = orders_df.groupby('product')['unit_price'].transform(lambda x: x.fillna(x.mean()))
        orphaned_orders = ~(orders_df['customer_id'].isin(customers_df['customer_id']))
        orphaned_orders_df = orders_df[orphaned_orders]
        print(f"{len(orphaned_orders_df)} orphaned orders rejected")
        orders_df = orders_df[~orphaned_orders]
        return customers_df, orders_df, orphaned_orders_df

In [ ]:
def load(df, table_name, engine):
    try:
        with engine.connect() as conn:
                conn.execute(text(f"TRUNCATE TABLE {table_name}"))
                conn.commit()     
        df.to_sql(table_name, con=engine, if_exists="append", index=False)
        print(f'Loaded {len(df)} rows into {table_name}')
    except OperationalError:
        print('Database connection failed')
    except (IntegrityError, DatabaseError):
        print(f"{table_name} rows are duplicated")
    
    

In [ ]:
customers_df, orders_df = extract(r'C:\Users\HP\Downloads\customers.csv', r'C:\Users\HP\Downloads\orders.csv')
customers_df, orders_df, orphaned_orders_df = clean(customers_df, orders_df)
load(orders_df, "orders", engine)
load(customers_df, "customers", engine)

In [ ]:
try:
    load(customers_df, "customers", engine)
except Exception as e:
    print(type(e))
    print(type(e).__mro__)

In [ ]:
customers_df = pd.read_csv(r'C:\Users\HP\Downloads\customers.csv')

In [ ]:
orders_df = pd.read_csv(r'C:\Users\HP\Downloads\orders.csv')

In [ ]:
customers_df.info()

In [ ]:
customers_df.head()

In [ ]:
customers_df.isnull().sum()


In [ ]:
orders_df.head()

In [ ]:
orders_df.info()

In [ ]:
orders_df.isnull().sum()

In [ ]:
customers_df.duplicated().sum()

In [ ]:
orphaned_orders = ~(orders_df['customer_id'].isin(customers_df['customer_id']))

In [ ]:
orphaned_orders_df = orders_df[orphaned_orders]          

In [ ]:
orders_df = orders_df[~orphaned_orders]   

In [ ]:
print(len(orphaned_orders_df))
print(len(orders_df))

In [ ]:
customers_df = customers_df.drop_duplicates()

In [ ]:
print(customers_df.duplicated().sum())
len(customers_df)

In [ ]:
customers_df['email_missing'] = customers_df['email'].isnull()

In [ ]:
customers_df['email_missing'].sum()

In [ ]:
orders_df['unit_price'] = orders_df.groupby('product')['unit_price'].transform(lambda x: x.fillna(x.mean()))

In [ ]:
orders_df.isna().sum()

In [ ]:
customers_df['customer_name'] = customers_df['customer_name'].str.strip().str.title()

In [ ]:
customers_df['city'] = customers_df['city'].str.strip().str.title()
print(customers_df['city'].unique())
customers_df['city'] = customers_df['city'].replace({"Ph": "Port Harcourt"})
print(customers_df['city'].unique())

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

engine = create_engine(f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

In [ ]:
load(customers_df, 'customers', bad_engine)

In [ ]:
with engine.connect() as conn:
    print("Connected successfully")

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE orders, customers RESTART IDENTITY CASCADE"))
    conn.commit()

customers_df.to_sql('customers', engine, if_exists='append', index=False)
orders_df.to_sql('orders', engine, if_exists='append', index=False)

print("Load complete")

In [ ]:
customers_df.head()

In [ ]:
orders_df.head()

In [ ]:
orphaned_orders_df.head()

In [ ]:
print(orders_df['customer_id'].isin(customers_df['customer_id']).all())

In [ ]:

def extract(customers_path, orders_path):
    try:
        customers_df = pd.read_csv(customers_path)
        orders_df = pd.read_csv(orders_path)
        return customers_df, orders_df
    except FileNotFoundError:
        print('File not found')
        return None, None

In [ ]:
c, o = extract('nonexistent_file.csv', 'nonexistent_file.csv')
print(c, o)

In [ ]:
print(customers_df.head())
print(orders_df.head())

In [ ]:
def clean(customers_df, orders_df):
    customers_df['city'] = customers_df['city'].str.strip().str.title()
    customers_df['city'] = customers_df['city'].replace({"Ph": "Port Harcourt"})
    customers_df['customer_name'] = customers_df['customer_name'].str.strip().str.title()
    customers_df = customers_df.drop_duplicates()
    customers_df['email_missing'] = customers_df['email'].isnull()
    orders_df['unit_price'] = orders_df.groupby('product')['unit_price'].transform(lambda x: x.fillna(x.mean()))
    orphaned_orders = ~(orders_df['customer_id'].isin(customers_df['customer_id']))
    orphaned_orders_df = orders_df[orphaned_orders] 
    orders_df = orders_df[~orphaned_orders]   
    return customers_df, orders_df, orphaned_orders_df

In [ ]:
customers_df, orders_df, orphaned_orders_df = clean(customers_df, orders_df)

In [ ]:
print(customers_df['city'].unique())
print(customers_df.isnull().sum())
print(len(orphaned_orders_df))